# SAM2 Floor Plan Fine-tuning in Colab

This notebook guides you through fine-tuning SAM2 on your floor plan dataset in Google Colab.

## Prerequisites

- Google Colab with GPU (A100 80GB recommended)
- Your floor plan dataset uploaded to Google Drive or Colab
- Dataset structure: `tuning_dataset/images/floorplan1/00000.png` through `tuning_dataset/images/floorplan30/00000.png`
- Annotations: `tuning_dataset/annotations/floorplan1/00000.png` through `tuning_dataset/annotations/floorplan30/00000.png`


## Step 1: Mount Google Drive (if your data is on Drive)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# If your data is on Drive, set this path
# DATA_PATH = '/content/drive/MyDrive/path/to/tuning_dataset'
# Otherwise, upload data directly to Colab and set:
# DATA_PATH = '/content/tuning_dataset'


## Step 2: Install SAM2 and Dependencies


In [ ]:
# Clone SAM2 repository
!git clone https://github.com/facebookresearch/sam2.git
%cd sam2  # This puts us in the repo root (where setup.py is)

# Install SAM2 with training dependencies
!pip install -e ".[dev]"

# Verify installation
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")


## Step 3: Download SAM2.1 Checkpoint


In [ ]:
# Create checkpoints directory
!mkdir -p checkpoints

# Download SAM2.1 Base Plus checkpoint
!wget -O checkpoints/sam2.1_hiera_base_plus.pt https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_base_plus.pt

print("Checkpoint downloaded successfully!")


## Step 4: Set Up Dataset Paths


In [ ]:
# Set your dataset path here
DATA_PATH = '/content/drive/MyDrive/tuning_dataset'  # Update this path!

# Or if uploaded directly to Colab:
# DATA_PATH = '/content/tuning_dataset'

import os
if not os.path.exists(DATA_PATH):
    print(f"⚠️ Warning: Dataset path {DATA_PATH} does not exist!")
    print("Please update DATA_PATH to point to your tuning_dataset directory")
else:
    print(f"✅ Dataset found at: {DATA_PATH}")


## Step 5: Validate Dataset (Optional)


In [ ]:
# Run validation script
!python scripts/validate_floorplan_data.py {DATA_PATH}


## Step 6: Create Train/Val Split


In [ ]:
# Create train/val split (24 train, 6 val)
!python scripts/create_train_val_split.py {DATA_PATH} --seed 42

# Verify split files were created
import os
if os.path.exists(f"{DATA_PATH}/train_list.txt") and os.path.exists(f"{DATA_PATH}/val_list.txt"):
    print("✅ Train/val split created successfully!")
    print("\nTrain samples:")
    !head -5 {DATA_PATH}/train_list.txt
    print("\nVal samples:")
    !head -5 {DATA_PATH}/val_list.txt
else:
    print("❌ Error creating split files")


## Step 7: Update Training Configuration


In [ ]:
# Update the config file with your dataset paths and checkpoint path
# Note: If the config file doesn't exist yet, you'll need to create it from the MOSE example
# or copy it from your local repo. For now, we assume it exists.

config_path = 'sam2/configs/sam2.1_training/sam2.1_hiera_b+_floorplan_finetune.yaml'

# Get absolute path for checkpoint (relative to current working directory)
import os
current_dir = os.getcwd()
checkpoint_path = os.path.join(current_dir, 'checkpoints', 'sam2.1_hiera_base_plus.pt')

# Check if config exists, if not, create it from template
if not os.path.exists(config_path):
    print("⚠️ Config file not found. Creating from MOSE template...")
    # You may need to manually create this file or copy it from your repo
    print("Please ensure the config file exists at:", config_path)
else:
    # Read the file
    with open(config_path, 'r') as f:
        content = f.read()
    
    # Replace the paths
    content = content.replace('img_folder: null', f'img_folder: {DATA_PATH}/images')
    content = content.replace('gt_folder: null', f'gt_folder: {DATA_PATH}/annotations')
    content = content.replace('file_list_txt: null', f'file_list_txt: {DATA_PATH}/train_list.txt')
    # Update checkpoint path (handle both null and relative path cases)
    if 'checkpoint_path: null' in content:
        content = content.replace('checkpoint_path: null', f'checkpoint_path: {checkpoint_path}')
    elif './checkpoints/sam2.1_hiera_base_plus.pt' in content:
        content = content.replace('./checkpoints/sam2.1_hiera_base_plus.pt', checkpoint_path)
    
    # Write back
    with open(config_path, 'w') as f:
        f.write(content)
    
    print("✅ Configuration updated!")
    print(f"   Image folder: {DATA_PATH}/images")
    print(f"   Annotation folder: {DATA_PATH}/annotations")
    print(f"   Train list: {DATA_PATH}/train_list.txt")
    print(f"   Checkpoint: {checkpoint_path}")
    
    # Verify checkpoint exists
    if os.path.exists(checkpoint_path):
        print(f"   ✅ Checkpoint found!")
    else:
        print(f"   ⚠️  Checkpoint not found at {checkpoint_path}")
        print(f"   Make sure you've downloaded it in Step 3!")


## Step 8: Start Training


In [ ]:
# Launch training
# This will train for 40 epochs by default
# Training logs and checkpoints will be saved to sam2_logs/

!python training/train.py \
    -c sam2/configs/sam2.1_training/sam2.1_hiera_b+_floorplan_finetune.yaml \
    --use-cluster 0 \
    --num-gpus 1


## Step 9: Monitor Training with TensorBoard


In [ ]:
# Load TensorBoard extension
%load_ext tensorboard

# Start TensorBoard (adjust log_dir if needed)
# The log directory is typically: sam2_logs/sam2.1_hiera_b+_floorplan_finetune/tensorboard
%tensorboard --logdir sam2_logs


## Step 10: Save Checkpoint to Drive (Optional)

After training completes, save your checkpoint to Google Drive for later use.


In [ ]:
# Find the latest checkpoint
import glob
checkpoint_dir = 'sam2_logs/sam2.1_hiera_b+_floorplan_finetune/checkpoints'
checkpoints = glob.glob(f'{checkpoint_dir}/*.pt')

if checkpoints:
    latest_checkpoint = max(checkpoints, key=os.path.getctime)
    print(f"Latest checkpoint: {latest_checkpoint}")
    
    # Copy to Drive (update path as needed)
    drive_checkpoint_path = '/content/drive/MyDrive/sam2_floorplan_checkpoint.pt'
    !cp {latest_checkpoint} {drive_checkpoint_path}
    print(f"✅ Checkpoint saved to: {drive_checkpoint_path}")
else:
    print("No checkpoints found. Training may still be in progress.")


## Notes

- Training time: ~2-4 hours on A100 80GB GPU for 40 epochs
- Monitor GPU memory usage - if you get OOM errors, reduce `train_batch_size` in the config
- The final checkpoint will be saved in `sam2_logs/sam2.1_hiera_b+_floorplan_finetune/checkpoints/`
- You can use the fine-tuned checkpoint just like the original SAM2 checkpoints for inference
